this script doesn't run because there is no dataset linked to it

# Introduction

After running a single-trial analysis like a regression, decoding or RSA, we end up with custom single-trial stcs that instead of containing normalized amplitude z-scores (dSPM) contain our variable of interest: β values, ρ values etc.

In order to establish group level significance, we follow a two-stage analysis pipeline like the one mentioned in [Gwilliams, Lewis and Marantz](https://https://www.sciencedirect.com/science/article/pii/S105381191600166X) (2016) where we take these newly formed STCs of our variables of interest and run a univariate test. In this tutorial, we'll also be applying spatio-temporal clustering to correct for multiple comparisons.

Most of the time, you'll end up running your analyses in different conditions and have a custom STC object per condition/model per subject...

**IMPORTANT**: this notebook only allows you to establish group-level significance, any subject level significance needs to be determined before running these tests.


---



# Code
## Settings


In [ ]:
import os, sys
import numpy as np
import mne
from pylab import *
import eelbrain as e
from functools import reduce
from pathlib import Path
import operator

### Define important variables


In [ ]:
e.configure(n_workers=True, prompt_toolkit="eelbrain")

PATH = Path("Experiment_Path")
main_dir = PATH / "data"  # dataset directory
mri_dir = main_dir / "mri"
meg_dir = main_dir / "meg"
log_dir = PATH / "logs"

os.chdir(PATH) # set the working directory

# set subjects
subjects = [d for d in os.listdir(meg_dir) if "R" in d and ".md" not in d]

# set sampling rate if downsampled data
fs = 500

# set clustering threshold
pval = 0.05

### ROI

The test we'll be running is going to be within a specific ROI. Since we want to be able to compare across subjects we'll be loading in labels from Fsaverage, an average MRI that allows us to compare each subject's activity in a common space.

We'll define an ROI in two ways:
- a simple ROI that corresponds to one region
- a custom label

Here we're using Brodmann Areas to define a custom label

In [ ]:
# create custom ROI label (ex: AG + STG)
hemi = 'lh'
annot_name = "PALS_B12_Brodmann"

# read in annotation file
brodmann = mne.read_labels_from_annot(
        "fsaverage", annot_name, subjects_dir=mri_dir, hemi=hemi
    )

# create a list of the regions to load from the file
# here BA 40, BA 39 and BA 22
name_list = [f"Brodmann.40-{hemi}", f"Brodmann.39-{hemi}", f"Brodmann.22-{hemi}"]
label_list = [label for label in brodmann if label.name in name_list]

# this step takes the list of labels we've loaded and just adds them together
agstg = reduce(operator.add, label_list)

Now let's use the Desikan-Killiany atlas (aparc) to isolate one region

In [ ]:
# create a simple ROI label (ex: precuneus)
hemi = 'lh'
annot_name = "aparc"
    aparc = mne.read_labels_from_annot(
        "fsaverage", annot_name, subjects_dir=mri_dir, hemi=hemi
    ) # we've loaded in another annotation file that has more specific anatomical labels

name = f"precuneus-{hemi}"
pre = [label for label in aparc if label.name == name][0]

## Script

### Create eelbrain dataset

We start by loading in the individual subject files from our single trial analysis...

In [ ]:
# load in single trial stcs as a list or a dict depending on the type of analysis
# requires loading in saved data
# any normalizing step that needs to happen can be plugged in here
results = []
print("Done loading results!")

now iterate through the stcs and extract the subject as well as morphing/normalizing data if needed

In [ ]:
# create eelbrain dataset
morphed_vals, subjectslist = [], []
for stc in results:
    # morph the results if single trial analysis not on morphed stcs
    # if no morphing just load the stcs into a list to create the final eelbrain dataset
    subj = stc.subject
    print(f"Morphing subject {subj}...")
    morph = mne.compute_source_morph(
        stc,
        subject_from=subj,
        subject_to="fsaverage",
        subjects_dir=mri_dir,
        spacing=4,
        )

    # morph to the fsaverage brain
    stc = morph.apply(stc)
    print(f"STCs morphed")
    morphed_vals.append(stc)
    subjectslist.append(subj)

print("Full dataset created and morphed!")

Finally, create the eelbrain dataset:
Eelbrain datasets are like dictionaries so we can create columns with data the way we'd define a dict or pandas dataframe

In [ ]:
ds = e.Dataset()
ds["subject"] = e.Factor(subjectslist, random=True)  # within subj
ds["src"] = e.load.fiff.stc_ndvar(
    morphed_vals, "fsaverage", "ico-4", mri_dir
) # load in the source space

# get ROI
src = ds["src"] # select the source space

### 7: Temporal clustering

We start by setting our source space to the ROI and then averaging. What this means is that we are collapsing over space to find timepoints where the average activity in the cluster is significant.

In [ ]:
roi = src.sub(source=pre)
# average over the source dimension
temproi = roi.mean('source')
# here we are choosing to not add the roi back to the ds but you can define a new column

print("STCs subset to area of interest. Starting one sample t-test!")

There are many different tests you can run in order to establish significance... A **one sample t-test** is the most straight-forward as this helps you determine if the distribution of your results is significantly different from 0 (or 1, 0.5 etc.)

You can also run an ANOVA as a form of group-level model comparison, or a t-test between two results to see if two results are truly different.

eelbrain implements a limited range of tests BUT you could always implement custom testing and clustering yourself.

There are a couple of settings to play with when running tests:
- tstart: start of analysis window
- tstop: end of analysis window
- mintime: minimum duration of clusters to be considered for permutation
- samples: how many permutations
- pmin: minimum uncorrected p-value for a cluster to be included in permutations

In [ ]:
# Actual test
temp_res = e.testnd.TTestOneSample(
        temproi, # roi
        ds=ds, # dataset
        # samples=1000,  # number of permutations
        pmin=pval,  # threshold for clusters (uncorrected p-value)
        tstart=0,  # start of the time window of interest
        tstop=0.65,  # stop of the time window of interest
        mintime=0.40,  # minimum duration for clusters
        match= 'subject' # if multiple stcs for one subject match subject
        )

temp_res.clusters.sort("p")
temp_table = temp_res.clusters.as_table()
print('results for temporal clustering test:')
print(temp_table)

**STOP HERE for tutorial 7**

### 9: Spatio-temporal clustering

Code-wise the only difference between temporal and spatio-temporal clustering is that the input to the eelbrain function is no longer an average of activity over sources but is a full STC...
Conceptually, your clusters will now be bundles of activation in both space and time.

We also need to add one parameter: *minsource* which tells eelbrain how many sources a cluster needs to have in order to be included.
The higher the number the stricter the test...

In [ ]:
# Actual test
res = e.testnd.TTestOneSample(
        roi, # roi
        ds=ds, # dataset
        # samples=1000,  # number of permutations
        pmin=pval,  # threshold for clusters (uncorrected p-value)
        tstart=0,  # start of the time window of interest
        tstop=0.65,  # stop of the time window of interest
        mintime=0.40,  # minimum duration for clusters
        minsource=40,
        match= 'subject' # if multiple stcs for one subject match subject
        )

res.clusters.sort("p")
table = res.clusters.as_table()
print('results for temporal clustering test:')
print(table)